[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/int/u1_area.ipynb)

# Int. U1 · Área bajo la curva
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 1 del bloque de cálculo integral. Tiene una sección por subtema, en el mismo orden que el libro:

1. Áreas de figuras conocidas: área bajo una poligonal con trapecios
2. Aproximación con rectángulos: sumas inferior y superior
3. Área como acumulación: de la velocidad a la posición

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `INT-U1` (bloque, unidad). Las celdas de código de cada sección vienen de `int/u1_area/NN_*.py`; los fragmentos que el libro imprime, de `int/u1_area/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Las tablas se escriben como números separados por comas, por ejemplo `0, 1, 3, 4`. Para escribir funciones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)`, `exp(...)` y `log(...)` (natural). Los ángulos de `sin`, `cos` y `tan` van en **radianes**.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log,
              "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "E": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que los resultados salgan exactos
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)


def parsear(texto):
    """Convierte un texto en una expresión de sympy en x o explica qué salió mal."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una función, por ejemplo: x**2")
    local = {**_FUNCIONES, "x": x}
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x como variable y, si hace falta, "
                         "sqrt, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la función. Usa * para multiplicar, ** o ^ para potencias "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr) or expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Escribe una expresión en x, sin el signo = y solo con las funciones permitidas.")
    if expr.free_symbols - {x}:
        raise ValueError("Solo puedes usar la variable x.")
    return expr


def lista_numeros(texto, nombre="la lista"):
    """'0, 1.5, 3' -> [0.0, 1.5, 3.0], o un mensaje claro si algo no es número."""
    partes = [p for p in re.split(r"[,\s;]+", str(texto).strip()) if p]
    if not partes:
        raise ValueError(f"Escribe {nombre} como números separados por comas, por ejemplo: 0, 1, 3, 4")
    try:
        return [float(p) for p in partes]
    except ValueError as err:
        raise ValueError(f"En {nombre} hay algo que no es un número: usa punto decimal y comas entre valores.") from err


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() >= n:                     # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False

## 1. Áreas de figuras conocidas: área bajo una poligonal

Si la gráfica de $f\geq0$ está formada por segmentos de recta, la región bajo ella se parte en **trapecios**, uno por cada par de puntos consecutivos $(x_k,y_k)$ y $(x_{k+1},y_{k+1})$:

$$A_k=\frac{y_k+y_{k+1}}{2}\,(x_{k+1}-x_k).$$

Un rectángulo es un trapecio con lados paralelos iguales y un triángulo, uno con un lado paralelo de longitud cero. Por **aditividad**, el área total es la suma de las $A_k$. Las unidades del área son (unidad del eje $y$)·(unidad del eje $x$).

In [ ]:
def area_poligonal(xs, ys):
    """Áreas de los trapecios bajo la poligonal que une (xs[k], ys[k]) y su suma.
    Exige al menos dos puntos, x estrictamente creciente y y >= 0."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    if xs.size != ys.size:
        raise ValueError(f"hay {xs.size} valores de x y {ys.size} de y: deben ser los mismos.")
    if xs.size < 2:
        raise ValueError("se necesitan al menos dos puntos para formar un trapecio.")
    if np.any(np.diff(xs) <= 0):
        raise ValueError("los valores de x deben ir de menor a mayor, sin repetirse.")
    if np.any(ys < 0):
        raise ValueError("hay valores de y negativos: esta sección mide el área de regiones sobre el eje. "
                         "Para áreas con signo usa la calculadora de la sección 3.")
    areas = (ys[:-1] + ys[1:]) / 2 * np.diff(xs)
    return areas, float(areas.sum())


def calculadora_poligonal(x_txt, y_txt, unidad_x, unidad_y, n_cifras):
    try:
        xs, ys = lista_numeros(x_txt, "los valores de x"), lista_numeros(y_txt, "los valores de y")
        areas, total = area_poligonal(xs, ys)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    if unidad_x and unidad_x == unidad_y:
        u = f" {unidad_x}²"
    elif unidad_x or unidad_y:
        u = " " + "·".join(v for v in (unidad_y, unidad_x) if v)
    else:
        u = ""
    for k, a in enumerate(areas):
        print(f"  trapecio {k + 1}: x de {cifras(xs[k], n_cifras)} a {cifras(xs[k + 1], n_cifras)} -> área {cifras(a, n_cifras)}{u}")
    print(f"área total ≈ {cifras(total, n_cifras)}{u}   ({n_cifras} cifras significativas, redondeado)")
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.fill_between(xs, 0, ys, color="0.85")
    ax.plot(xs, ys, "o-", color="navy")
    for xv in xs[1:-1]:
        ax.axvline(xv, color="gray", ls="--", lw=0.7)
    ax.axhline(0, color="black", lw=0.7)
    ax.set_xlabel(f"x ({unidad_x})" if unidad_x else "x"); ax.set_ylabel(f"y ({unidad_y})" if unidad_y else "y")
    ax.set_title("Área bajo la poligonal"); plt.show()


widgets.interact(calculadora_poligonal,
    x_txt=widgets.Text(value="0, 1, 3, 4", description="x ="),
    y_txt=widgets.Text(value="1, 3, 3, 0", description="y ="),
    unidad_x=widgets.Text(value="", description="unidad x"),
    unidad_y=widgets.Text(value="", description="unidad y"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("rectángulo: f = 3 en [1, 5] da 12", lambda: cerca(area_poligonal([1, 5], [3, 3])[1], 12)),
    ("trapecio: f = 2x + 1 en [0, 3] da 12", lambda: cerca(area_poligonal([0, 3], [1, 7])[1], 12)),
    ("poligonal de la figura 1.1(c): 2 + 6 + 1.5 = 9.5",
     lambda: np.allclose(area_poligonal([0, 1, 3, 4], [1, 3, 3, 0])[0], [2, 6, 1.5])),
    ("|x - 2| en [0, 5]: dos triángulos, 6.5", lambda: cerca(area_poligonal([0, 2, 5], [2, 0, 3])[1], 6.5)),
    ("x decreciente o y negativa se rechazan", lambda: _rechaza(lambda: area_poligonal([0, 2, 1], [1, 1, 1]))
                                                     and _rechaza(lambda: area_poligonal([0, 1], [1, -1]))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano el área bajo la poligonal que une $(0,0)$, $(2,4)$, $(5,4)$ y $(6,0)$. Escribe tu resultado y ejecuta la celda.

In [ ]:
mi_area = None           # escribe un número, por ejemplo: 12.5

if mi_area is None:      # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = area_poligonal([0, 2, 5, 6], [0, 4, 4, 0])[1]
    print("La calculadora da:", cifras(ref, 4))
    print("coinciden" if cerca(mi_area, ref, 1e-6) else
          "NO coinciden: parte la región en un triángulo, un rectángulo y otro triángulo")

## 2. Aproximación con rectángulos: sumas inferior y superior

Partición regular: $\Delta x=\dfrac{b-a}{n}$, $x_k=a+k\,\Delta x$. En cada subintervalo $[x_{k-1},x_k]$, $m_k$ es el **mínimo** de $f$ y $M_k$ el **máximo**:

$$s_n=\sum m_k\,\Delta x\ \ (\text{inscritos})\qquad\le\qquad A\qquad\le\qquad S_n=\sum M_k\,\Delta x\ \ (\text{circunscritos}).$$

Si $f$ es **monótona**, el mínimo y el máximo están en los extremos de cada subintervalo y $S_n-s_n=|f(b)-f(a)|\,\Delta x$. Si **no** lo es, pueden estar dentro: la calculadora busca también los puntos donde $f'=0$.

In [ ]:
def _valor(f, c):
    v = complex(sp.N(f.subs(x, c)))
    if abs(v.imag) > 1e-12 or not math.isfinite(v.real):
        raise ValueError(f"f no tiene valor real en x = {cifras(float(c), 6)}: revisa el dominio.")
    return v.real


def extremos_en(f, a, b):
    """(mínimo, máximo) de f en [a, b]: extremos del subintervalo y puntos interiores con f' = 0."""
    cand = [sp.nsimplify(a), sp.nsimplify(b)]
    crit = sp.solveset(sp.diff(f, x), x, domain=sp.Interval.open(sp.nsimplify(a), sp.nsimplify(b)))
    if isinstance(crit, sp.FiniteSet):
        cand += list(crit)
    elif crit is not sp.S.EmptySet:              # sympy no pudo listarlos: muestreo fino como respaldo
        cand += list(np.linspace(float(a), float(b), 401))
    vals = [_valor(f, c) for c in cand]
    return min(vals), max(vals)


def sumas_inf_sup(f, a, b, n):
    """Suma inferior, suma superior y lista de (m_k, M_k) con partición regular de n subintervalos."""
    if not a < b:
        raise ValueError("el intervalo debe cumplir a < b.")
    intervalo = sp.Interval(sp.nsimplify(a), sp.nsimplify(b))
    if not intervalo.is_subset(sp.calculus.util.continuous_domain(f, x, intervalo)):
        raise ValueError("f no es continua en todo [a, b] (tiene un salto, una asíntota o sale de su dominio); "
                         "las sumas inferior y superior de esta sección suponen f continua.")
    if int(n) != n or n < 1:
        raise ValueError("n debe ser un entero positivo.")
    n = int(n)
    xs = [sp.nsimplify(a) + k * (sp.nsimplify(b) - sp.nsimplify(a)) / n for k in range(n + 1)]
    dx = float(xs[1] - xs[0])
    mM = [extremos_en(f, xs[k], xs[k + 1]) for k in range(n)]
    if min(m for m, _ in mM) < -1e-12:
        raise ValueError("f toma valores negativos en [a, b]; esta sección trabaja con regiones sobre el eje.")
    return sum(m for m, _ in mM) * dx, sum(M for _, M in mM) * dx, mM


def monotona(f, a, b):
    """True si f' no cambia de signo en (a, b) (revisión en 400 puntos)."""
    d = sp.lambdify(x, sp.diff(f, x), "numpy")
    v = np.asarray(d(np.linspace(a, b, 402)[1:-1]), dtype=float) * np.ones(400)
    return bool(np.all(v >= -1e-12) or np.all(v <= 1e-12))


def n_para_tolerancia(f, a, b, tol):
    """Rectángulos que garantizan S_n - s_n <= tol si f es monótona (propiedad 1.6b)."""
    if not monotona(f, a, b):
        raise ValueError("f no es monótona en [a, b]: la fórmula |f(b) - f(a)|(b - a)/n no aplica.")
    salto = abs(_valor(f, b) - _valor(f, a)) * (b - a)
    return max(1, math.ceil(salto / tol - 1e-12))


def calculadora_rectangulos(f_txt, a, b, n, n_cifras, tolerancia):
    try:
        f = parsear(f_txt)
        s, S, mM = sumas_inf_sup(f, a, b, n)
    except (ValueError, TypeError) as err:
        print("Revisa la entrada:", err); return
    print(f"s_{n} ≈ {cifras(s, n_cifras)}   S_{n} ≈ {cifras(S, n_cifras)}   diferencia ≈ {cifras(S - s, n_cifras)}"
          f"   promedio ≈ {cifras((s + S) / 2, n_cifras)}   ({n_cifras} cifras significativas, redondeado)")
    try:
        print(f"para S_n - s_n <= {tolerancia}: n >= {n_para_tolerancia(f, a, b, tolerancia)}")
    except ValueError as err:
        print("tolerancia:", err)
    fx = sp.lambdify(x, f, "numpy")
    xx = np.linspace(a, b, 400)
    fig, ax = plt.subplots(figsize=(5.5, 3))
    dx = (b - a) / n
    for k, (m, M) in enumerate(mM):
        ax.add_patch(plt.Rectangle((a + k * dx, 0), dx, M, fill=False, hatch="///", edgecolor="gray"))
        ax.add_patch(plt.Rectangle((a + k * dx, 0), dx, m, color="0.8"))
    ax.plot(xx, np.asarray(fx(xx), dtype=float) * np.ones_like(xx), color="navy")
    ax.axhline(0, color="black", lw=0.7)
    ax.set_xlabel("x"); ax.set_ylabel("f(x)")
    ax.set_title(f"inscritos (gris) y circunscritos (achurado), n = {n}"); plt.show()


widgets.interact(calculadora_rectangulos,
    f_txt=widgets.Text(value="x**2", description="f(x) ="),
    a=widgets.FloatText(value=0, description="a"), b=widgets.FloatText(value=1, description="b"),
    n=widgets.IntSlider(value=4, min=1, max=60, description="n"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."),
    tolerancia=widgets.FloatText(value=0.01, description="tolerancia"));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
_s, _S, _ = sumas_inf_sup(x**2, 0, 1, 4)
_si, _Si, _ = sumas_inf_sup(sp.sin(x), 0, sp.pi, 3)
PRUEBAS_2 = [
    ("x^2 en [0, 1], n = 4: s = 7/32 y S = 15/32", lambda: cerca(_s, 7 / 32) and cerca(_S, 15 / 32)),
    ("1/x en [1, 2], n = 4: s ≈ 0.63452, S ≈ 0.75952",
     lambda: cerca(sumas_inf_sup(1 / x, 1, 2, 4)[0], 0.634523809, 1e-8) and cerca(sumas_inf_sup(1 / x, 1, 2, 4)[1], 0.759523809, 1e-8)),
    ("constante 3 en [0, 2], n = 5: s = S = 6", lambda: cerca(sumas_inf_sup(3 + 0 * x, 0, 2, 5)[0], 6)
                                                    and cerca(sumas_inf_sup(3 + 0 * x, 0, 2, 5)[1], 6)),
    ("sin x en [0, π], n = 3: máximo interior; s ≈ 0.9069, S ≈ 2.861",
     lambda: cerca(_si, 0.906899682, 1e-8) and cerca(_Si, 2.860996915, 1e-8)),
    ("x^2 en [0, 1]: S_n - s_n <= 0.001 exige n >= 1000", lambda: n_para_tolerancia(x**2, 0, 1, 0.001) == 1000),
    ("a >= b, f negativa o f discontinua se rechazan", lambda: _rechaza(lambda: sumas_inf_sup(x, 2, 1, 4))
                                                and _rechaza(lambda: sumas_inf_sup(x - 2, 0, 1, 4))
                                                and _rechaza(lambda: sumas_inf_sup(1 / (x - sp.Rational(1, 2))**2, 0, 1, 3))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano $s_4$ y $S_4$ para $f(x)=x$ en $[0,2]$. Comprueba que el área exacta, que conoces por geometría, queda entre tus dos sumas.

In [ ]:
mi_s4 = None     # escribe un número
mi_S4 = None     # escribe un número

if mi_s4 is None or mi_S4 is None:      # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    s4, S4, _ = sumas_inf_sup(x, 0, 2, 4)
    print("La calculadora da: s_4 =", cifras(s4, 4), " S_4 =", cifras(S4, 4), " (área exacta: triángulo de base 2 y altura 2)")
    print("coinciden" if cerca(mi_s4, s4, 1e-6) and cerca(mi_S4, S4, 1e-6) else
          "NO coinciden: con Δx = 0.5, la suma inferior usa 0, 0.5, 1 y 1.5 como alturas")

## 3. Área como acumulación: de la velocidad a la posición

Con $v(t)$ dada por una tabla y unida por rectas:

* **Desplazamiento** (área con signo): $\Delta s=A_+-A_-$. La posición final es $s_0+\Delta s$.
* **Distancia recorrida** (área total): $d=A_++A_-$, el área bajo $|v|$. Siempre $d\geq|\Delta s|$.

Si la velocidad cambia de signo entre dos lecturas, el cruce por cero está en $t_c=t_k+\dfrac{v_k\,(t_{k+1}-t_k)}{v_k-v_{k+1}}$ y cada lado se trata por separado. La misma calculadora sirve para cualquier tasa: caudal → volumen, corriente → carga, potencia → energía.

In [ ]:
def integrar_tabla(ts, vs, s0=0.0):
    """Desplazamiento, distancia, posición máxima y mínima y la tabla de posiciones (t, s)
    para una tasa v(t) definida por puntos y unida por rectas."""
    ts, vs = np.asarray(ts, dtype=float), np.asarray(vs, dtype=float)
    if ts.size != vs.size or ts.size < 2:
        raise ValueError("se necesitan al menos dos puntos y el mismo número de tiempos que de valores.")
    if np.any(np.diff(ts) <= 0):
        raise ValueError("los tiempos deben ir de menor a mayor, sin repetirse.")
    puntos = [(ts[0], vs[0])]
    for k in range(ts.size - 1):                       # inserta los cruces por cero
        t0, t1, v0, v1 = ts[k], ts[k + 1], vs[k], vs[k + 1]
        if v0 * v1 < 0:
            puntos.append((t0 + v0 * (t1 - t0) / (v0 - v1), 0.0))
        puntos.append((t1, v1))
    tp, vp = np.array(puntos).T
    trozos = (vp[:-1] + vp[1:]) / 2 * np.diff(tp)     # trapecios con signo; ninguno cruza el eje
    s = s0 + np.concatenate([[0.0], np.cumsum(trozos)])
    return {"desplazamiento": float(trozos.sum()), "distancia": float(np.abs(trozos).sum()),
            "A+": float(trozos[trozos > 0].sum()), "A-": float(-trozos[trozos < 0].sum()),
            "s_max": float(s.max()), "s_min": float(s.min()), "t": tp, "s": s, "v": vp}


def unidad_area(unidad_v, unidad_t):
    """(unidad del área, aviso). Si la tasa es 'algo/tiempo' y el tiempo coincide, el área queda en 'algo'."""
    unidad_v, unidad_t = unidad_v.strip(), unidad_t.strip()
    if "/" in unidad_v:
        num, den = (p.strip() for p in unidad_v.rsplit("/", 1))
        if den == unidad_t:
            return num, None
        return f"{unidad_v}·{unidad_t}", (f"la tasa está en {unidad_v} y el tiempo en {unidad_t}: convierte uno de los dos "
                                           "antes de interpretar el área (por ejemplo, minutos a horas).")
    return "·".join(u for u in (unidad_v, unidad_t) if u), None


def calculadora_velocidad(t_txt, v_txt, s0, unidad_t, unidad_v, n_cifras):
    try:
        r = integrar_tabla(lista_numeros(t_txt, "los tiempos"), lista_numeros(v_txt, "las velocidades"), s0)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    unidad_s, aviso = unidad_area(unidad_v, unidad_t)
    if aviso:
        print("AVISO:", aviso)
    u = f" {unidad_s}" if unidad_s else ""
    print(f"A+ ≈ {cifras(r['A+'], n_cifras)}{u}   A- ≈ {cifras(r['A-'], n_cifras)}{u}")
    print(f"desplazamiento ≈ {cifras(r['desplazamiento'], n_cifras)}{u}   posición final ≈ "
          f"{cifras(s0 + r['desplazamiento'], n_cifras)}{u}")
    print(f"distancia recorrida ≈ {cifras(r['distancia'], n_cifras)}{u}   posición máxima ≈ {cifras(r['s_max'], n_cifras)}{u}"
          f"   ({n_cifras} cifras significativas, redondeado)")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
    a1.fill_between(r["t"], 0, r["v"], where=r["v"] >= 0, color="0.8")
    a1.fill_between(r["t"], 0, r["v"], where=r["v"] <= 0, hatch="///", facecolor="white", edgecolor="gray")
    a1.plot(r["t"], r["v"], "o-", color="navy"); a1.axhline(0, color="black", lw=0.7)
    a1.set_xlabel(f"t ({unidad_t})"); a1.set_ylabel(f"tasa ({unidad_v})"); a1.set_title("áreas + y -")
    a2.plot(r["t"], r["s"], "o-", color="navy")
    a2.set_xlabel(f"t ({unidad_t})"); a2.set_ylabel(f"acumulado ({unidad_s})"); a2.set_title("acumulado")
    plt.tight_layout(); plt.show()


widgets.interact(calculadora_velocidad,
    t_txt=widgets.Text(value="0, 4", description="t ="),
    v_txt=widgets.Text(value="4, -4", description="v ="),
    s0=widgets.FloatText(value=0, description="s0"),
    unidad_t=widgets.Text(value="s", description="unidad t"),
    unidad_v=widgets.Text(value="m/s", description="unidad v"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
_agv = integrar_tabla([0, 2, 8, 10, 12, 13, 16, 17], [0, 1.2, 1.2, 0, 0, -0.8, -0.8, 0])
PRUEBAS_3 = [
    ("semilla: v = 2t en [0, 10] da 100 m", lambda: cerca(integrar_tabla([0, 10], [0, 20])["distancia"], 100)),
    ("v = 4 - 2t en [0, 4]: desplazamiento 0, distancia 8", lambda: cerca(integrar_tabla([0, 4], [4, -4])["desplazamiento"], 0)
                                                                  and cerca(integrar_tabla([0, 4], [4, -4])["distancia"], 8)),
    ("tanque: 3 L/s por 60 s y rampa a 0 en 30 s: 225 L", lambda: cerca(integrar_tabla([0, 60, 90], [3, 3, 0])["desplazamiento"], 225)),
    ("AGV: final 6.4, odómetro 12.8, máximo 9.6", lambda: cerca(_agv["desplazamiento"], 6.4) and cerca(_agv["distancia"], 12.8)
                                                       and cerca(_agv["s_max"], 9.6)),
    ("tiempos que no crecen se rechazan", lambda: _rechaza(lambda: integrar_tabla([0, 2, 2], [1, 1, 1]))),
    ("unidades: m/s por s da m; km/h por min avisa", lambda: unidad_area("m/s", "s") == ("m", None)
                                                         and unidad_area("km/h", "min")[1] is not None),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** La velocidad de un carrito pasa por $(0,2)$, $(3,2)$, $(4,-2)$ y $(6,-2)$ ($t$ en s, $v$ en m/s), unidos por rectas. Calcula a mano el desplazamiento y la distancia recorrida.

In [ ]:
mi_desplazamiento = None    # escribe un número
mi_distancia = None         # escribe un número

if mi_desplazamiento is None or mi_distancia is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    r = integrar_tabla([0, 3, 4, 6], [2, 2, -2, -2])
    print("La calculadora da: desplazamiento =", cifras(r["desplazamiento"], 4), "m   distancia =", cifras(r["distancia"], 4), "m")
    print("coinciden" if cerca(mi_desplazamiento, r["desplazamiento"], 1e-6) and cerca(mi_distancia, r["distancia"], 1e-6) else
          "NO coinciden: la velocidad cruza el cero en t = 3.5 s; separa los dos triángulos que se forman ahí")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")